# 03 · Review, kiểm định và độ nhạy
Đọc review đã chọn một dòng/đơn và cờ is_review_valid từ mart. Main dùng review điểm 1–5 tạo cùng/sau ngày giao. Review trước giao chỉ dùng kiểm tra độ nhạy. Các kiểm định thể hiện liên quan, không suy ra nhân quả. Chạy **Restart Kernel → Run All** sau SQL 08.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'sql/07_create_mart_views.sql').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the project or notebooks folder.')
sys.path.insert(0, str(ROOT / 'notebooks'))
from utils.db import get_engine, read_mart
START, END = pd.Timestamp('2017-01-01'), pd.Timestamp('2018-09-01')
EXPORT_RESULTS = False  # Set True to save the displayed tables and figures.
OUT = ROOT / 'data/exports/sql_mart'
tables, figures = {}, {}
def cohort(frame):
    return frame.loc[frame.order_purchase_timestamp.ge(START)
                     & frame.order_purchase_timestamp.lt(END)].copy()

In [ ]:
engine = get_engine()
try:
    orders = cohort(read_mart('logistics', engine))
    categories = cohort(read_mart('category', engine))
finally:
    engine.dispose()
assert orders.order_id.is_unique and not orders.empty
assert not categories.duplicated(['order_id','category_en']).any()
eligible = orders[orders.is_eligible.eq(1)].copy()
assert not eligible.empty and eligible.is_late.isin([0,1]).all()
assert eligible.loc[eligible.is_review_valid.eq(1), 'review_score'].between(1,5).all()

## Review theo mức trễ và coverage
Coverage = số review hợp lệ / eligible; low-review rate = điểm 1–2 / review hợp lệ. Nhóm không có review có low-review rate NULL, không phải 0.

In [ ]:
eligible['delay_band'] = np.select([eligible.is_late.eq(0), eligible.delay_days.le(3),
    eligible.delay_days.le(7)], ['on_time','late_1_3','late_4_7'], default='late_8_plus')
def review_summary(frame, groups, strict=True):
    data = frame.copy()
    data['valid'] = data.is_review_valid.eq(1) if strict else data.review_score.between(1,5)
    data['low'] = data['valid'] & data.review_score.le(2)
    result = data.groupby(groups, dropna=False).agg(eligible_orders=('order_id','size'),
        reviews=('valid','sum'), low_reviews=('low','sum')).reset_index()
    result['coverage'] = result.reviews / result.eligible_orders
    result['low_review_rate'] = result.low_reviews / result.reviews.replace(0,np.nan)
    return result
review_groups = review_summary(eligible,['is_late'])
review_bands = review_summary(eligible,['delay_band'])
tables.update(review_groups=review_groups,review_bands=review_bands)
display(review_groups);display(review_bands)
fig,ax=plt.subplots(figsize=(7,4))
ax.bar(review_groups.is_late.map({0:'On time',1:'Late'}),review_groups.low_review_rate*100)
ax.set(ylabel='% valid reviews',title='Low review rate: reviews on/after delivery')
fig.tight_layout();plt.show();figures['review_groups']=fig

## Mann–Whitney và kích thước tác động
Kiểm định hai phía trên điểm review, một quan sát/đơn. Rank-biserial = 2U/(n_late × n_on_time) − 1; âm nghĩa nhóm trễ có điểm thấp hơn. Báo cả cỡ mẫu và coverage để nhìn thấy selection bias.

In [ ]:
from scipy.stats import mannwhitneyu
valid=eligible[eligible.is_review_valid.eq(1)]
x=valid.loc[valid.is_late.eq(1),'review_score'].astype(float)
y=valid.loc[valid.is_late.eq(0),'review_score'].astype(float)
u=p_value=effect=np.nan
if len(x) and len(y):
    u,p_value=mannwhitneyu(x,y,alternative='two-sided')
    effect=2*u/(len(x)*len(y))-1
test=pd.DataFrame([{'late_n':len(x),'on_time_n':len(y),'mann_whitney_u':u,
    'p_value':p_value,'rank_biserial_late_minus_on_time':effect}])
tables['review_test']=test;display(test)

## Kiểm tra độ nhạy và phân tầng
So sánh bỏ tháng 2018-08, dùng mọi timing review, bỏ outlier theo cờ SQL. Cờ is_duration_outlier là định nghĩa trong warehouse, không tính lại ngưỡng ở Python. Kết quả phân tầng theo bang/ngành là mô tả, chưa phải hiệu ứng đã điều chỉnh.

In [ ]:
scenarios = [('main',eligible,True),
    ('exclude_august',eligible[eligible.order_purchase_timestamp.lt('2018-08-01')],True),
    ('all_review_timing',eligible,False),
    ('exclude_duration_outliers',eligible[eligible.is_duration_outlier.eq(0)],True)]
sensitivity=[]
for name,frame,strict in scenarios:
    table=review_summary(frame,['is_late'],strict)
    table['scenario']=name
    table['late_rate']=frame.is_late.mean()
    table['delivery_sample']=frame.delivery_days.count()
    table['median_days']=frame.delivery_days.median()
    table['p90_days']=frame.delivery_days.quantile(.9)
    table['median_delay']=frame.delay_days.median()
    table['p90_delay']=frame.delay_days.quantile(.9)
    sensitivity.append(table)
sensitivity=pd.concat(sensitivity,ignore_index=True)
strata=review_summary(categories[categories.is_eligible.eq(1)],['category_en','customer_state','is_late'])
tables.update(review_sensitivity=sensitivity,review_strata=strata)
display(sensitivity);display(strata.head(30))

## Ghi kết luận vào báo cáo
Ghi rõ phạm vi, mẫu số, coverage và bằng chứng bảng/biểu đồ. Dùng từ “liên quan”; không coi p-value nhỏ là tác động lớn hay chứng minh nguyên nhân. Cập nhật reports/insights_log.md và case study thủ công sau khi đọc các kết quả mới.

In [ ]:
if EXPORT_RESULTS:
    OUT.mkdir(parents=True, exist_ok=True)
    for name, table in tables.items():
        table.to_csv(OUT / f'{name}.csv', index=False, encoding='utf-8-sig')
    for name, figure in figures.items():
        figure.savefig(OUT / f'{name}.png', dpi=160, bbox_inches='tight')
    print(f'Exported SQL mart results to {OUT}')
else:
    print('Results displayed only. Set EXPORT_RESULTS=True and Run All to export.')